Hi! I am a Kaggle beginner and not fluent in English, so this notebook is written in Japanese.

Please use your browser's translation (or DeepL/ChatGPT) to read it!

Hope it helps!

# データの読み込み・ライブラリのインポート

In [1]:
import pandas as pd
import numpy as np

import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

import kagglehub
import os

In [2]:
path = kagglehub.competition_download("playground-series-s6e11")

train = pd.read_csv(os.path.join(path, "train.csv"))
test = pd.read_csv(os.path.join(path, "test.csv"))
sample_submission = pd.read_csv(os.path.join(path, "sample_submission.csv"))

## データの確認

In [3]:
train.head()

,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,...,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt,is_fraud
0,0,61.16,9.77,1,17.78,48,56,1269.35,0,5.5,...,OTP,POS,iPhone,False,False,False,False,False,False,0.0
1,1,24.83,1.54,3,13.06,38,28,3233.00,0,33.4,...,PIN,In-App,Android Phone,False,False,False,False,False,False,0.0
2,2,6.36,1.20,4,8.17,50,55,1046.88,0,35.8,...,OTP,Online,Tablet,False,False,True,False,False,False,0.0
3,3,37.44,5.13,4,8.67,48,48,2217.67,0,20.7,...,3D Secure,POS,Android Phone,False,False,False,False,False,False,0.0
4,4,653.85,0.80,4,11.69,37,52,3291.06,0,34.8,...,3D Secure,Online,Windows PC,False,False,False,False,False,False,0.0


In [4]:
test.head()

,id,amount_usd,hours_since_last_txn,txn_count_last_24h,distance_from_home_km,card_age_months,customer_age,account_balance_usd,cvv_retry_count,velocity_score,...,card_type,auth_method,channel,device_type,is_foreign_transaction,is_new_merchant,used_vpn,ip_country_mismatch,billing_shipping_mismatch,is_ai_generated_scam_attempt
0,699960,28.21,4.52,4,17.87,37,45,1483.49,0,15.4,...,Visa,OTP,Online,iPhone,False,False,False,False,False,False
1,699961,644.90,5.48,1,6.68,52,28,2183.64,0,11.2,...,RuPay,PIN,Contactless,Android Phone,False,False,False,False,False,False
2,699962,20.53,12.20,3,15.39,44,75,1208.59,0,20.8,...,Visa,No Authentication,Online,iPhone,False,False,False,False,False,False
3,699963,10.83,6.90,2,44.34,50,54,1790.85,0,9.2,...,Mastercard,PIN,Contactless,Mac,False,False,False,False,False,False
4,699964,19.00,0.63,3,25.22,50,66,1676.18,0,20.0,...,Mastercard,PIN,In-App,Android Phone,False,False,False,False,False,False


In [5]:
sample_submission.head()

,id,is_fraud
0,699960,0.0167
1,699961,0.0167
2,699962,0.0167
3,699963,0.0167
4,699964,0.0167


In [6]:
print(f'train の件数 = {len(train)}')
print(f'test の件数 = {len(test)}')
print(f'sample_submission の件数 = {len(sample_submission)}')

train の件数 = 699960
test の件数 = 299984
sample_submission の件数 = 299984


In [7]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699960 entries, 0 to 699959
Data columns (total 26 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   id                            699960 non-null  int64  
 1   amount_usd                    699960 non-null  float64
 2   hours_since_last_txn          699960 non-null  float64
 3   txn_count_last_24h            699960 non-null  int64  
 4   distance_from_home_km         699960 non-null  float64
 5   card_age_months               699960 non-null  int64  
 6   customer_age                  699960 non-null  int64  
 7   account_balance_usd           699960 non-null  float64
 8   cvv_retry_count               699960 non-null  int64  
 9   velocity_score                699960 non-null  float64
 10  time_of_day_hour              699960 non-null  int64  
 11  day_of_week                   699960 non-null  int64  
 12  merchant_risk_score           699960 non-nul

## 各列の意味

参考：[https://www.kaggle.com/competitions/playground-series-s6e11/data?select=train.csv](http://)

- `id`
- `amount_usd`
- `hours_since_last_txn`
- `txn_count_last_24h`  
- `distance_from_home_km` : 家からの距離
- `card_age_months` : カードの利用歴（月） 
- `customer_age` : 顧客の年齢  
- `account_balance_usd`
- `cvv_retry_count` : リトライ回数  
- `velocity_score`
- `time_of_day_hour`  
- `day_of_week`  
- `merchant_risk_score` : 加盟店リスクスコア
- `prior_disputes` : 過去の紛争
- `merchant_category` : オンライン小売 22% / 食料品 20% / その他 57%
- `card_type` : Visa 49% / Mastercard 36% / その他 14% 
- `auth_method` : 3Dセキュア 30% / ワンタイムパスワード (OTP) 25% / その他 45%
- `channel` : オンライン 39% / POS 29% / その他 31% 
- `device_type` : Android 34% / iPhone 24% / Other 43% 
- `is_foreign_transaction`  : 外国取引 (True 2% / False 98%) 
- `is_new_merchant` : 新規加盟店 (True 15% / False 85%) 
- `used_vpn` :  (True 3% / False 97%) 
- `ip_country_mismatch` : IPアドレスの国情報不一致 (True 2% / False 98%) 
- `billing_shipping_mismatch` : 請求先・配送先の不一致 (True 1% / False 99%) 
- `is_ai_generated_scam_attempt` AI による詐欺行為であるか (True 0% / False 100%) 
- `is_fraud` : 不正の有無 **(目的変数)**

# 関数の定義

In [8]:
def train_lgb(train_df, test_df, feature_cols, target_col='is_fraud'):
    """
    指定された特徴量で LightGBM の交差検証学習を行い、
    OOF予測値（確率）、テスト予測値（確率）、学習済みモデル群を返す関数
    """
    
    # bool型をそのまま int（1/0）に変換
    y_train = train_df[target_col].astype(int).values
    
    # ループの前にカテゴリ型への一括変換
    X_train_df = train_df[feature_cols].copy()
    X_test_df = test_df[feature_cols].copy()
    
    for col in feature_cols:
        if X_train_df[col].dtype == 'object':
            X_train_df[col] = X_train_df[col].astype('category')
            X_test_df[col] = X_test_df[col].astype('category')

    X_train_df = X_train_df.reset_index(drop=True)

    cv = KFold(n_splits=5, shuffle=True, random_state=0)
    
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'n_jobs': -1,
        'random_state': 0,
        'verbose': -1,
        'enable_categorical': True,
    }

    models = []
    oof_train = np.zeros(len(X_train_df))
    
    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train_df, y_train)):
        X_tr = X_train_df.iloc[train_idx]
        y_tr = y_train[train_idx]
        X_va = X_train_df.iloc[valid_idx]
        y_va = y_train[valid_idx]

        # Datasetの作成 (free_raw_data=True でメモリを節約)
        lgb_train = lgb.Dataset(X_tr, y_tr, free_raw_data=True)
        lgb_eval = lgb.Dataset(X_va, y_va, reference=lgb_train, free_raw_data=True)

        model = lgb.train(
            params,
            lgb_train,
            valid_sets=[lgb_eval],
            num_boost_round=1000,
            callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
        )

        best_iter = model.best_iteration
        best_score = model.best_score['valid_0']['auc']
        print(f'Fold {fold+1} | Best Iteration: {best_iter:4d} | Best AUC: {best_score:.5f}')

        oof_train[valid_idx] = model.predict(X_va, num_iteration=model.best_iteration)
        
        models.append(model)

    # 全体の OOF ROC-AUC スコアを計算
    score = roc_auc_score(y_train, oof_train)
    print(f'\nOOF ROC-AUC Score: {score:.5f}')

    # すべての学習が終わった後、ループの外でテストデータを一括予測する
    y_preds = [model.predict(X_test_df, num_iteration=model.best_iteration) for model in models]
    y_preds_mean = np.mean(y_preds, axis=0)

    return oof_train, y_preds_mean, models

# 実行

In [9]:
%%time

feature_cols = [
    'amount_usd',
    'hours_since_last_txn',
    'txn_count_last_24h',  
    'distance_from_home_km',
    'card_age_months', 
    'customer_age',
    'account_balance_usd',
    'cvv_retry_count', 
    'velocity_score',
    'time_of_day_hour',
    'day_of_week', 
    'merchant_risk_score',
    'prior_disputes',
    'merchant_category',
    'card_type',
    'auth_method', 
    'channel',
    'device_type',
    'is_foreign_transaction',  
    'is_new_merchant',  
    'used_vpn',  
    'ip_country_mismatch',
    'billing_shipping_mismatch', 
    'is_ai_generated_scam_attempt'  
]


# 関数の実行
oof, preds, models = train_lgb(train, test, feature_cols)


# 特徴量重要度の可視化（表形式で出力）
feature_importances = np.mean(
    [model.feature_importance(importance_type='gain') for model in models],
    axis=0,
)

importance_df = (
    pd.DataFrame({'feature': feature_cols, 'importance': feature_importances})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

display(importance_df.head(20))

Fold 1 | Best Iteration:   90 | Best AUC: 0.94293
Fold 2 | Best Iteration:   68 | Best AUC: 0.94816
Fold 3 | Best Iteration:   87 | Best AUC: 0.93989
Fold 4 | Best Iteration:   86 | Best AUC: 0.94384
Fold 5 | Best Iteration:   71 | Best AUC: 0.94606

OOF ROC-AUC Score: 0.94406


,feature,importance
0,merchant_risk_score,52869.524579
1,cvv_retry_count,45247.593414
2,velocity_score,33699.304677
3,ip_country_mismatch,26569.959459
4,auth_method,24993.515200
5,is_new_merchant,20910.659472
6,customer_age,11665.513806
7,billing_shipping_mismatch,10211.430209
8,used_vpn,7583.651303
9,is_foreign_transaction,5701.442425


CPU times: user 2min 45s, sys: 652 ms, total: 2min 46s
Wall time: 43.7 s
